# Mushroom Classification with ViT


## 1. Налаштування середовища

In [1]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ GPU не знайдено! Перейдіть в Runtime → Change runtime type → GPU")

CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
Memory: 85.1 GB


In [2]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive підключено")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive підключено


In [3]:

import os
os.chdir('/content')

if os.path.exists('Applied_Mashroomatics'):
    os.chdir('Applied_Mashroomatics')
    !git pull
else:
    !git clone https://github.com/konstanta-asya/Applied_Mashroomatics.git
    os.chdir('Applied_Mashroomatics')

print(f"Робоча директорія: {os.getcwd()}")

Already up to date.
Робоча директорія: /content/Applied_Mashroomatics


In [4]:
!pip install -q tqdm timm

In [14]:
!rm -rf /content/DF20M /content/DF20M-metadata
!cp -r /content/drive/MyDrive/raw/DF20M /content/
!cp -r /content/drive/MyDrive/raw/DF20M-metadata /content/

In [15]:
import os

# === КОПІЮВАННЯ ДАНИХ ЛОКАЛЬНО (швидше тренування) ===
DRIVE_DATA = '/content/drive/MyDrive/raw/DF20M'
DRIVE_META = '/content/drive/MyDrive/raw/DF20M-metadata'
LOCAL_DATA = '/content/DF20M'
LOCAL_META = '/content/DF20M-metadata'

if not os.path.exists(LOCAL_DATA):
    print("Копіюємо зображення локально (~10-15 хв)...")
    !cp -r {DRIVE_DATA} {LOCAL_DATA}
    print(f"Готово! {len(os.listdir(LOCAL_DATA))} файлів")
else:
    print(f"Дані вже є: {len(os.listdir(LOCAL_DATA))} файлів")

if not os.path.exists(LOCAL_META):
    print("Копіюємо metadata...")
    !cp -r {DRIVE_META} {LOCAL_META}

# Symlinks для скрипта
!rm -rf data/raw && mkdir -p data/raw
!ln -s {LOCAL_DATA} data/raw/DF20M
!ln -s {LOCAL_META} data/raw/DF20M-metadata

print("Дані готові!")

Дані вже є: 36430 файлів
Дані готові!


## 2. Тренування моделі

**ViT + Metadata Fusion:**
- Модель: `vit_base_patch16_224` з metadata токеном
- Metadata: habitat, substrate, month (sinusoidal encoding)
- Backbone заморожений перші 5 епох, потім розморожується
- Differential LR: backbone=3e-5, meta/head=1e-4

**Оптимізації:**
- Mixed precision (AMP)
- Label smoothing (0.1)
- Gradient clipping (max_norm=1.0)
- Early stopping (patience=5)
- **RESUME** - продовжує з останнього checkpoint

In [17]:
# === METADATA FUSION TRAINING (A100) ===
BATCH_SIZE = 128
EPOCHS = 30
NUM_WORKERS = 4
RESUME = True  # True якщо продовжуємо

resume_flag = f"--resume /content/drive/MyDrive/mushroom_checkpoints/latest_checkpoint.pth" if RESUME else ""

!python src/train_vit.py \
    --csv_path data/raw/DF20M-metadata/DF20M-train_metadata_PROD.csv \
    --data_dir data/raw/DF20M \
    --batch_size {BATCH_SIZE} \
    --epochs {EPOCHS} \
    --amp \
    --num_workers {NUM_WORKERS} \
    --checkpoint_dir /content/drive/MyDrive/mushroom_checkpoints \
    {resume_flag}

Using device: cuda
CONFIG: lr=3e-05, meta_lr=0.0001, weight_decay=0.05, epochs=30
Backbone frozen for first 5 epochs
Mixed precision (AMP) enabled
Built vocabularies:
  Habitat: 30 categories
  Substrate: 24 categories
Number of classes: 179
Train batches: 205, Val batches: 52
Habitat vocab size: 30, Substrate vocab size: 24
Creating MushroomViTWithMetadata (vit_base_patch16_224)
Backbone FROZEN for initial training
Creating optimizer with differential LR:
  Param group 'meta_encoder': 221,952 params, lr=1.00e-04
  Param group 'head': 139,187 params, lr=1.00e-04
  Param group 'meta_token': 768 params, lr=1.00e-04
Resuming from checkpoint: /content/drive/MyDrive/mushroom_checkpoints/latest_checkpoint.pth
  Param group 'backbone': 85,798,656 params, lr=3.00e-05
  Param group 'meta_encoder': 221,952 params, lr=1.00e-04
  Param group 'head': 139,187 params, lr=1.00e-04
  Param group 'meta_token': 768 params, lr=1.00e-04
/content/Applied_Mashroomatics/src/train_vit.py:115: UserWarning: Dete

In [22]:
import torch
import torch.nn.functional as F
from PIL import Image
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from torchvision import transforms
from tqdm import tqdm

# Завантажити модель
checkpoint = torch.load('/content/drive/MyDrive/mushroom_checkpoints/best_model.pth')

from src.models.mushroom_vit import MushroomViTWithMetadata
model = MushroomViTWithMetadata(
    num_classes=checkpoint['num_classes'],
    habitat_vocab=checkpoint['habitat_vocab'],
    substrate_vocab=checkpoint['substrate_vocab'],
    pretrained=False
)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.cuda().eval()

habitat_vocab = checkpoint['habitat_vocab']
substrate_vocab = checkpoint['substrate_vocab']

# Дані — ТОЧНО як в data_setup.py
df = pd.read_csv('data/raw/DF20M-metadata/DF20M-train_metadata_PROD.csv')
df = df.dropna(subset=['species']).reset_index(drop=True)

species_list = sorted(df['species'].unique())
species_to_id = {sp: i for i, sp in enumerate(species_list)}

unique_ids = df['gbifID'].unique()
_, val_ids = train_test_split(unique_ids, test_size=0.2, random_state=42)
val_df = df[df['gbifID'].isin(val_ids)].reset_index(drop=True)

print(f"Val samples: {len(val_df)}")
print(f"Classes: {len(species_to_id)}")

# TTA transforms
tta_transforms = [
    transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.CenterCrop(224),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    transforms.Compose([
        transforms.Resize((288, 288)),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
]

correct = 0
total = 0

with torch.no_grad():
    for idx in tqdm(range(len(val_df)), desc="TTA"):
        row = val_df.iloc[idx]
        img_path = os.path.join('data/raw/DF20M', row['image_path'])

        try:
            image = Image.open(img_path).convert('RGB')
        except:
            continue

        label = species_to_id[row['species']]

        h_col = 'Habitat' if 'Habitat' in val_df.columns else 'habitat'
        s_col = 'Substrate' if 'Substrate' in val_df.columns else 'substrate'

        h_val = row.get(h_col)
        s_val = row.get(s_col)

        h_id = habitat_vocab.get(h_val, len(habitat_vocab)) if pd.notna(h_val) else len(habitat_vocab)
        s_id = substrate_vocab.get(s_val, len(substrate_vocab)) if pd.notna(s_val) else len(substrate_vocab)
        month = int(row['month']) if pd.notna(row.get('month')) else 6

        all_probs = []
        for tf in tta_transforms:
            img_tensor = tf(image).unsqueeze(0).cuda()
            h = torch.tensor([h_id]).cuda()
            s = torch.tensor([s_id]).cuda()
            m = torch.tensor([month]).cuda()

            output = model(img_tensor, h, s, m)
            probs = F.softmax(output, dim=1)
            all_probs.append(probs)

        avg_probs = torch.stack(all_probs).mean(dim=0)
        pred = avg_probs.argmax(1).item()

        total += 1
        correct += (pred == label)

print(f"\n{'='*50}")
print(f"Val Accuracy (no TTA): 71.44%")
print(f"Val Accuracy (TTA x3): {100.*correct/total:.2f}%")
print(f"{'='*50}")



Val samples: 6582
Classes: 179


TTA: 100%|██████████| 6582/6582 [08:37<00:00, 12.72it/s]


Val Accuracy (no TTA): 71.44%
Val Accuracy (TTA x3): 72.29%


In [20]:
from PIL import Image
import os
import pandas as pd
from torchvision import transforms
import torch.nn.functional as F

# TTA transforms
tta_transforms = [
    # Original
    transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    # HFlip
    transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.CenterCrop(224),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    # Larger crop
    transforms.Compose([
        transforms.Resize((288, 288)),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
]

# Load val data info
df = pd.read_csv('data/raw/DF20M-metadata/DF20M-train_metadata_PROD.csv')
from sklearn.model_selection import train_test_split
unique_ids = df['gbifID'].unique()
_, val_ids = train_test_split(unique_ids, test_size=0.2, random_state=42)
df_clean = df.dropna(subset=['species'])
species_list = sorted(df_clean['species'].unique())
species_to_id = {sp: i for i, sp in enumerate(species_list)}

val_df = df_clean[df_clean['gbifID'].isin(val_ids)].reset_index(drop=True)


habitat_vocab = checkpoint['habitat_vocab']
substrate_vocab = checkpoint['substrate_vocab']


correct = 0
total = 0

with torch.no_grad():
    for idx in tqdm(range(len(val_df)), desc="TTA"):
        row = val_df.iloc[idx]
        img_path = os.path.join('data/raw/DF20M', row['image_path'])

        try:
            image = Image.open(img_path).convert('RGB')
        except:
            continue

        label = species_to_id[row['species']]
        h_id = habitat_vocab.get(row.get('Habitat'), len(habitat_vocab))
        s_id = substrate_vocab.get(row.get('Substrate'), len(substrate_vocab))
        month = int(row['month']) if pd.notna(row.get('month')) else 6

        # TTA: average predictions
        all_probs = []
        for tf in tta_transforms:
            img_tensor = tf(image).unsqueeze(0).cuda()
            h = torch.tensor([h_id]).cuda()
            s = torch.tensor([s_id]).cuda()
            m = torch.tensor([month]).cuda()

            output = model(img_tensor, h, s, m)
            probs = F.softmax(output, dim=1)
            all_probs.append(probs)

        avg_probs = torch.stack(all_probs).mean(dim=0)
        pred = avg_probs.argmax(1).item()

        total += 1
        correct += (pred == label)

print(f"\nTTA Accuracy (3 augments): {100.*correct/total:.2f}%")


TTA: 100%|██████████| 6449/6449 [08:08<00:00, 13.19it/s]


TTA Accuracy (3 augments): 91.75%


In [21]:
print(f"Val samples in TTA: {len(val_df)}")
print(f"Val batches in training: 52 * 128 = {52*128}")

# Перевір чи species_to_id співпадає з тренуванням
print(f"Num classes in checkpoint: {checkpoint['num_classes']}")
print(f"Num classes in species_to_id: {len(species_to_id)}")


Val samples in TTA: 6449
Val batches in training: 52 * 128 = 6656
Num classes in checkpoint: 179
Num classes in species_to_id: 179


## 3. Результати

In [7]:
import os

checkpoint_dir = '/content/drive/MyDrive/mushroom_checkpoints'

if os.path.exists(checkpoint_dir):
    print("📁 Збережені файли:")
    for f in os.listdir(checkpoint_dir):
        size = os.path.getsize(os.path.join(checkpoint_dir, f)) / 1e6
        print(f"  - {f} ({size:.1f} MB)")
else:
    print("❌ Папка з чекпоінтами не знайдена")

📁 Збережені файли:
  - best_model.pth (1034.1 MB)
  - latest_checkpoint.pth (1034.1 MB)


In [8]:

from google.colab import files
import os

best_model = '/content/drive/MyDrive/mushroom_checkpoints/best_model.pth'

if os.path.exists(best_model):
    files.download(best_model)
    print("✅ Модель завантажено")
else:
    print("❌ Файл best_model.pth не знайдено. Тренування ще не завершено?")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Модель завантажено


## 4. Тестування моделі

In [23]:
import torch
import sys
sys.path.insert(0, '/content/Applied_Mashroomatics')

from src.models.mushroom_vit import MushroomViTWithMetadata

checkpoint = torch.load('/content/drive/MyDrive/mushroom_checkpoints/best_model.pth')
num_classes = checkpoint['num_classes']
habitat_vocab = checkpoint['habitat_vocab']
substrate_vocab = checkpoint['substrate_vocab']

print(f"Model type: {checkpoint.get('model_type', 'unknown')}")
print(f"Кількість класів: {num_classes}")
print(f"Точність на валідації: {checkpoint['val_acc']:.2f}%")
print(f"Habitat categories: {len(habitat_vocab)}")
print(f"Substrate categories: {len(substrate_vocab)}")

model = MushroomViTWithMetadata(
    num_classes=num_classes,
    habitat_vocab=habitat_vocab,
    substrate_vocab=substrate_vocab,
    pretrained=False
)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print("Модель завантажено і готова до використання")

Model type: mushroom_vit_metadata
Кількість класів: 179
Точність на валідації: 71.44%
Habitat categories: 30
Substrate categories: 24
Модель завантажено і готова до використання
